In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/abhishekroy48/youtube-data/processed_youtube_global.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/reference.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/AIMO3_Reference_Problems.pdf
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/sample_submission.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/aimo_3_inference_server.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/aimo_3_gateway.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/__init__.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/templates.py
/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/kaggle_evaluation/core/base_gateway.py
/kaggle/input/competitions/ai-mathematic

In [ ]:
# /kaggle/input/datasets/abhishekroy48/youtube-data/processed_youtube_global.csv

In [ ]:
!pip install -q "FlagEmbedding==1.2.11" "transformers==4.44.2" faiss-cpu pandas numpy pyarrow

# Generate Embedings

In [1]:
# !pip install -q FlagEmbedding transformers torch faiss-gpu pandas numpy pyarrow

import os
import numpy as np
import pandas as pd
from pathlib import Path
from FlagEmbedding import BGEM3FlagModel
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


2026-03-22 22:05:47.499917: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774217147.621056     341 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774217147.659962     341 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774217147.971294     341 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774217147.971321     341 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774217147.971323     341 computation_placer.cc:177] computation placer alr

CUDA available: True
GPU: NVIDIA H100 80GB HBM3


In [3]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
MASTER_CSV  = "/kaggle/input/datasets/abhishekroy48/youtube-data/processed_youtube_global.csv"
EMBED_OUT   = "embeddings_bge_m3.npy"
META_OUT    = "embeddings_meta.parquet"
BATCH_SIZE  = 64        # H100 can safely go 128
MAX_SEQ_LEN = 512
# ─────────────────────────────────────────────────────────────────────────────

df = pd.read_csv(MASTER_CSV)
print(f"Loaded {len(df):,} rows | countries: {sorted(df['country_code'].unique())}")

df = df[df["mega_string"].notna() & (df["mega_string"].str.strip() != "")]
df = df.reset_index(drop=True)
print(f"After cleaning: {len(df):,} rows ready for embedding")
df.head(2)


Loaded 24,499 rows | countries: ['CA', 'DE', 'FR', 'GB', 'IN', 'JP', 'KR', 'MX', 'RU', 'US']
After cleaning: 24,499 rows ready for embedding


,video_id,trending_date,title,channel_title,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,trending_frequency,category_name,velocity_score,is_verified,mega_string,country_code
0,WG8JK-cPu-g,17.01.12,At Home with Amy Sedaris - Amy's Not-So Holida...,truTV,2017-11-27T20:30:03.000Z,"truTV|""tru""|""funny because its tru""|""truTV tru...",3597,59,8,13,https://i.ytimg.com/vi/WG8JK-cPu-g/default.jpg,1,Entertainment,0.8927,True,Title: At Home with Amy Sedaris - Amy's Not-So...,US
1,CV0J3Bq3BIc,17.01.12,Black Mirror - Black Museum | Official Trailer...,Netflix,2017-11-29T11:00:03.000Z,"Netflix|""Trailer""|""Netflix Original Series""|""N...",391639,7307,84,594,https://i.ytimg.com/vi/CV0J3Bq3BIc/default.jpg,1,Entertainment,1.5944,True,Title: Black Mirror - Black Museum | Official ...,US


In [5]:
model = BGEM3FlagModel(
    "BAAI/bge-m3",
    use_fp16=True,
    device="cuda" if torch.cuda.is_available() else "cpu",
)
print("Model loaded.")


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Model loaded.


In [ ]:
video_id,trending_date,title,channel_title,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,trending_frequency,category_name,velocity_score,is_verified,mega_string,country_code

In [6]:
texts = df["mega_string"].tolist()
print(f"Embedding {len(texts):,} texts in batches of {BATCH_SIZE}…")

all_embeddings = []
for i in range(0, len(texts), BATCH_SIZE):
    batch = texts[i : i + BATCH_SIZE]
    out   = model.encode(
        batch,
        batch_size=BATCH_SIZE,
        max_length=MAX_SEQ_LEN,
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False,
    )
    all_embeddings.append(out["dense_vecs"])
    if (i // BATCH_SIZE) % 20 == 0:
        print(f"  {i + len(batch):,} / {len(texts):,}")

embeddings = np.vstack(all_embeddings).astype("float32")
print(f"\nEmbedding matrix shape: {embeddings.shape}")

np.save(EMBED_OUT, embeddings)

meta_cols = ["video_id", "title", "channel_title", "category_name",
             "views", "likes", "velocity_score", "is_verified", "country_code"]
df[meta_cols].to_parquet(META_OUT, index=True)

print(f"Saved embeddings → {EMBED_OUT}")
print(f"Saved metadata   → {META_OUT}")


Embedding 24,499 texts in batches of 64…
  64 / 24,499
  1,344 / 24,499
  2,624 / 24,499
  3,904 / 24,499
  5,184 / 24,499
  6,464 / 24,499
  7,744 / 24,499
  9,024 / 24,499
  10,304 / 24,499
  11,584 / 24,499
  12,864 / 24,499
  14,144 / 24,499
  15,424 / 24,499
  16,704 / 24,499
  17,984 / 24,499
  19,264 / 24,499
  20,544 / 24,499
  21,824 / 24,499
  23,104 / 24,499
  24,384 / 24,499

Embedding matrix shape: (24499, 1024)
Saved embeddings → embeddings_bge_m3.npy
Saved metadata   → embeddings_meta.parquet


In [7]:
# ── Run this cell to reload saved artefacts without re-embedding ──────────────
embeddings = np.load(EMBED_OUT)
meta_df    = pd.read_parquet(META_OUT)

print(f"Embeddings : {embeddings.shape}")
print(f"Meta rows  : {len(meta_df):,}")
print(f"Countries  : {sorted(meta_df['country_code'].unique())}")


Embeddings : (24499, 1024)
Meta rows  : 24,499
Countries  : ['CA', 'DE', 'FR', 'GB', 'IN', 'JP', 'KR', 'MX', 'RU', 'US']


# Recommendation Engine


In [9]:
import faiss

norms    = np.linalg.norm(embeddings, axis=1, keepdims=True)
emb_norm = (embeddings / norms).astype("float32")

DIM = emb_norm.shape[1]   # 1024

def make_index(vecs):
    idx = faiss.IndexFlatIP(DIM)
    idx.add(vecs)
    return idx

index_global = make_index(emb_norm)
print(f"Global index: {index_global.ntotal:,} vectors")

country_indexes  = {}
country_row_maps = {}

for cc in meta_df["country_code"].unique():
    mask      = meta_df["country_code"] == cc
    global_ix = meta_df.index[mask].tolist()
    vecs      = emb_norm[global_ix]
    cidx      = make_index(vecs)
    country_indexes[cc]  = cidx
    country_row_maps[cc] = global_ix
    print(f"  [{cc}] {cidx.ntotal:,} vectors")

print("\nAll indexes built.")

Global index: 24,499 vectors
  [US] 2,440 vectors
  [CA] 2,398 vectors
  [DE] 2,579 vectors
  [FR] 2,514 vectors
  [GB] 1,835 vectors
  [IN] 2,508 vectors
  [JP] 2,658 vectors
  [KR] 2,557 vectors
  [MX] 2,501 vectors
  [RU] 2,509 vectors

All indexes built.


In [17]:
# ── CONFIGURE HERE ────────────────────────────────────────────────────────────
USER_COUNTRY = "US"        # home country of the user

# Replace with real video_ids from your dataset
WATCHED_VIDEO_IDS = [
    "CV0J3Bq3BIc",         # example from US dataset — replace me
    "we2XODhx-CM",
    "k_aT3jAK_QU",
    "be3BHwNi58A"
]

# Bucket sizes
TOTAL_RECS  = 30
N_BUCKET_A  = round(TOTAL_RECS * 0.6)   # 13  — semantic, same country
N_BUCKET_B  = round(TOTAL_RECS * 0.2)   # 3   — semantic, best foreign country
N_BUCKET_C  = round(TOTAL_RECS * 0.10)   # 2   — trending, user's country
N_BUCKET_D  = TOTAL_RECS - N_BUCKET_A - N_BUCKET_B - N_BUCKET_C  # 2 — trending, global
# ─────────────────────────────────────────────────────────────────────────────

print(f"Bucket sizes  →  A:{N_BUCKET_A}  B:{N_BUCKET_B}  C:{N_BUCKET_C}  D:{N_BUCKET_D}  total:{TOTAL_RECS}")

# Resolve watched IDs → global row indices
watched_mask    = meta_df["video_id"].isin(WATCHED_VIDEO_IDS)
watched_indices = set(meta_df.index[watched_mask].tolist())

if not watched_indices:
    raise ValueError("None of the watched video IDs were found in the metadata. "
                     "Check WATCHED_VIDEO_IDS and make sure the IDs exist in your CSV.")

print(f"Resolved {len(watched_indices)}/{len(WATCHED_VIDEO_IDS)} watched videos:")
print(meta_df.loc[list(watched_indices), ["video_id", "title", "country_code","category_name","channel_title"]].to_string())


Bucket sizes  →  A:18  B:6  C:3  D:3  total:30
Resolved 8/4 watched videos:
          video_id                                                               title country_code    category_name                       channel_title
1      CV0J3Bq3BIc       Black Mirror - Black Museum | Official Trailer [HD] | Netflix           US    Entertainment                             Netflix
2      we2XODhx-CM  Chelsea Handler Questions Whether She's Cut Out for a Relationship           US    Entertainment                        TheEllenShow
3      k_aT3jAK_QU         Justin Timberlake And Stephen Harmonize The National Anthem           US    Entertainment  The Late Show with Stephen Colbert
4      be3BHwNi58A  Prince Harry and Meghan Markle to wed in Windsor in May - BBC News           US  News & Politics                            BBC News
9961   k_aT3jAK_QU         Justin Timberlake And Stephen Harmonize The National Anthem           GB    Entertainment  The Late Show with Stephen Colbert
4750  

In [18]:
# Mean-pool watched embeddings → single taste vector, then re-normalise
watched_vecs = emb_norm[list(watched_indices)]
taste_vec    = watched_vecs.mean(axis=0)
taste_vec   /= np.linalg.norm(taste_vec)
taste_vec    = taste_vec.reshape(1, -1).astype("float32")

print(f"Taste vector shape: {taste_vec.shape}  |  norm: {np.linalg.norm(taste_vec):.4f}")


Taste vector shape: (1, 1024)  |  norm: 1.0000


In [23]:
import random

def search_index(index, row_map, query, k, exclude_global_ids):
    fetch_k = k + len(exclude_global_ids) + 10
    scores, local_positions = index.search(query, min(fetch_k, index.ntotal))
    results = []
    for lpos, score in zip(local_positions[0], scores[0]):
        if lpos < 0:
            continue
        gpos = row_map[lpos] if row_map is not None else lpos
        if gpos not in exclude_global_ids:
            results.append((gpos, float(score)))
        if len(results) == k:
            break
    return results

# ── BUCKET A — semantic, same country ────────────────────────────────────────
bucket_a_raw = search_index(
    country_indexes[USER_COUNTRY],
    country_row_maps[USER_COUNTRY],
    taste_vec,
    N_BUCKET_A,
    watched_indices,
)
used_ids = watched_indices | {r[0] for r in bucket_a_raw}
print(f"Bucket A: {len(bucket_a_raw)} videos (semantic, {USER_COUNTRY})")

# ── BUCKET B — semantic, mixed foreign countries ──────────────────────────────
# Compute affinity scores for all foreign countries
foreign_countries = [cc for cc in country_indexes if cc != USER_COUNTRY]
country_affinity  = {}

for cc in foreign_countries:
    gix  = country_row_maps[cc]
    vecs = emb_norm[gix]
    sims = (vecs @ taste_vec.T).flatten()
    country_affinity[cc] = float(sims.mean())

affinity_str = "  |  ".join(f"{cc}: {v:.4f}" for cc, v in
                             sorted(country_affinity.items(), key=lambda x: -x[1]))
print(f"\nCountry affinity scores:\n  {affinity_str}")

# Build weights for probabilistic sampling
countries   = list(country_affinity.keys())
raw_weights = np.array([country_affinity[cc] for cc in countries])
weights     = raw_weights - raw_weights.min() + 1e-3
weights     = weights / weights.sum()

# Decide how many videos to pull from each foreign country
# e.g. N_BUCKET_B=3 → could be [CA:1, JP:1, KR:1] or [GB:2, IN:1] etc.
# We assign 1 slot per country, drawn probabilistically without replacement
n_countries_to_mix = min(N_BUCKET_B, len(countries))  # can't mix more than available
chosen_countries   = np.random.choice(
    countries,
    size=n_countries_to_mix,
    replace=False,          # no duplicate countries in same batch
    p=weights,
).tolist()

# Distribute slots — base 1 per country, give leftover slots to highest affinity
slots = {cc: 1 for cc in chosen_countries}
leftover = N_BUCKET_B - n_countries_to_mix
if leftover > 0:
    # Sort chosen countries by affinity descending, give extras to top ones
    for cc in sorted(chosen_countries, key=lambda x: -country_affinity[x])[:leftover]:
        slots[cc] += 1

print(f"Bucket B slot allocation: {slots}")

bucket_b_raw = []
for cc, n_slots in slots.items():
    results = search_index(
        country_indexes[cc],
        country_row_maps[cc],
        taste_vec,
        n_slots,
        used_ids,
    )
    bucket_b_raw.extend(results)
    used_ids |= {r[0] for r in results}
    print(f"  [{cc}] contributed {len(results)} videos")

print(f"Bucket B: {len(bucket_b_raw)} videos (semantic, mixed foreign)")

# ── BUCKET C — random sample from top-100 velocity, user's country ────────────
TRENDING_POOL = 50

c_mask   = (meta_df["country_code"] == USER_COUNTRY) & (~meta_df.index.isin(used_ids))
c_pool   = meta_df[c_mask].nlargest(TRENDING_POOL, "velocity_score")
c_sample = c_pool.sample(n=min(N_BUCKET_C, len(c_pool)), random_state=None)
bucket_c_raw = [(idx, row["velocity_score"]) for idx, row in c_sample.iterrows()]
used_ids |= {r[0] for r in bucket_c_raw}
print(f"Bucket C: {len(bucket_c_raw)} videos (trending pool={TRENDING_POOL}, {USER_COUNTRY})")

# ── BUCKET D — random sample from top-100 velocity, mixed countries ───────────
# Instead of pure global, sample from a MIX of countries so one dominant
# country doesn't flood the trending bucket either
d_mask        = ~meta_df.index.isin(used_ids)
d_pool_full   = meta_df[d_mask].nlargest(TRENDING_POOL * 3, "velocity_score")

# Pick top TRENDING_POOL but ensure no single country takes more than 40%
max_per_country = max(1, int(TRENDING_POOL * 0.4))
d_pool_balanced = (
    d_pool_full
    .groupby("country_code", group_keys=False)
    .apply(lambda g: g.nlargest(max_per_country, "velocity_score"))
    .nlargest(TRENDING_POOL, "velocity_score")
)

d_sample     = d_pool_balanced.sample(n=min(N_BUCKET_D, len(d_pool_balanced)), random_state=None)
bucket_d_raw = [(idx, row["velocity_score"]) for idx, row in d_sample.iterrows()]
print(f"Bucket D: {len(bucket_d_raw)} videos (trending pool={TRENDING_POOL}, mixed global)")
print(f"  Country mix in D: {d_sample['country_code'].value_counts().to_dict()}")

print(f"\nTotal unique recommendations: "
      f"{len(bucket_a_raw)+len(bucket_b_raw)+len(bucket_c_raw)+len(bucket_d_raw)}")

Bucket A: 18 videos (semantic, US)

Country affinity scores:
  GB: 0.5941  |  CA: 0.5783  |  FR: 0.5558  |  DE: 0.5551  |  IN: 0.5484  |  MX: 0.5447  |  RU: 0.5439  |  KR: 0.5395  |  JP: 0.5324
Bucket B slot allocation: {'GB': 1, 'CA': 1, 'RU': 1, 'FR': 1, 'IN': 1, 'DE': 1}
  [GB] contributed 1 videos
  [CA] contributed 1 videos
  [RU] contributed 1 videos
  [FR] contributed 1 videos
  [IN] contributed 1 videos
  [DE] contributed 1 videos
Bucket B: 6 videos (semantic, mixed foreign)
Bucket C: 3 videos (trending pool=50, US)
Bucket D: 3 videos (trending pool=50, mixed global)
  Country mix in D: {'DE': 2, 'MX': 1}

Total unique recommendations: 30


/tmp/ipykernel_341/123278804.py:106: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.nlargest(max_per_country, "velocity_score"))


In [24]:
def build_bucket_df(raw, bucket_label, score_col):
    """Turn a list of (global_idx, score) into a labelled DataFrame slice."""
    if not raw:
        return pd.DataFrame()
    rows          = meta_df.iloc[[r[0] for r in raw]].copy()
    rows[score_col] = [round(r[1], 4) for r in raw]
    rows["bucket"]  = bucket_label
    rows["score_col"] = score_col
    return rows.reset_index(drop=True)

df_a = build_bucket_df(bucket_a_raw, "A — Semantic / Same Region",    "cosine_similarity")
df_b = build_bucket_df(bucket_b_raw, "B — Semantic / Mixed Foreign",  "cosine_similarity")
df_c = build_bucket_df(bucket_c_raw, "C — Trending / User Region",    "velocity_score")
df_d = build_bucket_df(bucket_d_raw, "D — Trending / Mixed Global",   "velocity_score")

recs = pd.concat([df_a, df_b, df_c, df_d], ignore_index=True)

DISPLAY_COLS = ["video_id", "title", "channel_title",
                "category_name", "country_code",
                "velocity_score", "cosine_similarity"]

for col in ["cosine_similarity", "velocity_score"]:
    if col not in recs.columns:
        recs[col] = np.nan

# Summary of foreign countries that ended up in the mix
foreign_mix = (
    recs[recs["bucket"] == "B — Semantic / Mixed Foreign"]["country_code"]
    .value_counts().to_dict()
)
trending_mix = (
    recs[recs["bucket"] == "D — Trending / Mixed Global"]["country_code"]
    .value_counts().to_dict()
)

print(f"{'='*80}")
print(f"  RECOMMENDATIONS  |  home: {USER_COUNTRY}")
print(f"  Foreign semantic mix : {foreign_mix}")
print(f"  Global trending mix  : {trending_mix}")
print(f"{'='*80}\n")

for bucket_label, group in recs.groupby("bucket", sort=False):
    print(f"── {bucket_label} ({len(group)} videos) ──")
    print(group[DISPLAY_COLS].to_string(index=False))
    print()

  RECOMMENDATIONS  |  home: US
  Foreign semantic mix : {'GB': 1, 'CA': 1, 'RU': 1, 'FR': 1, 'IN': 1, 'DE': 1}
  Global trending mix  : {'DE': 2, 'MX': 1}

── A — Semantic / Same Region (18 videos) ──
   video_id                                                                    title                        channel_title   category_name country_code  velocity_score  cosine_similarity
Dc6ZDo9ViNA                 Clarice Probes Hannibal Lecter About Trump's Russia Ties   The Late Show with Stephen Colbert   Entertainment           US          8.9855             0.7634
Jiia2IcFwqk                          Sarah Jessica Parker Finally Gets Asked To Prom   The Late Show with Stephen Colbert   Entertainment           US          3.9105             0.7631
9fe3ECuYWlI                                Americans Try To Explain The Royal Family                        BuzzFeedVideo  People & Blogs           US          2.0259             0.7629
q11UD-6XT-8                    Stephen's Covetton House

# Populate the Database

In [25]:
!pip install -q supabase python-dotenv


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 730.3/730.3 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 112.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 13.9 MB/s eta 0:00:00


In [26]:
import os
import numpy as np
import pandas as pd
from supabase import create_client, Client

# ── YOUR SUPABASE CREDENTIALS ─────────────────────────────────────────────────
# Get these from: Supabase Dashboard → Project Settings → API
# Add them as Kaggle Secrets (Add-ons → Secrets) — never hardcode them here!
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

SUPABASE_URL  = secrets.get_secret("SUPABASE_URL")   # https://xxxx.supabase.co
SUPABASE_KEY  = secrets.get_secret("SUPABASE_KEY")   # service_role key (not anon)
# ─────────────────────────────────────────────────────────────────────────────

supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
print("Supabase client created.")
print(f"Project URL: {SUPABASE_URL}")


Supabase client created.
Project URL: https://gqpxcspldebemsbpfpfi.supabase.co


In [27]:
# Reload embeddings + metadata (skip if still in memory from embedding run)
EMBED_OUT = "embeddings_bge_m3.npy"
META_OUT  = "embeddings_meta.parquet"

embeddings = np.load(EMBED_OUT)          # (N, 1024) float32
meta_df    = pd.read_parquet(META_OUT)   # aligned row index

# Also reload the full CSV for the extra columns not in meta (tags, trending_date etc.)
MASTER_CSV = "/kaggle/input/datasets/abhishekroy48/youtube-data/processed_youtube_global.csv"
full_df    = pd.read_csv(MASTER_CSV)
full_df    = full_df[full_df["mega_string"].notna() & (full_df["mega_string"].str.strip() != "")]
full_df    = full_df.reset_index(drop=True)

print(f"Embeddings : {embeddings.shape}")
print(f"Meta rows  : {len(meta_df):,}")
print(f"Full CSV   : {len(full_df):,}")
assert len(embeddings) == len(meta_df) == len(full_df),     "Row counts don't match — re-run the embedding cell first!"
print("All row counts aligned ✓")


Embeddings : (24499, 1024)
Meta rows  : 24,499
Full CSV   : 24,499
All row counts aligned ✓


In [28]:
def build_record(idx: int) -> dict:
    """Build a single record dict ready for Supabase upsert."""
    row  = full_df.iloc[idx]
    vec  = embeddings[idx].tolist()   # convert np.array → plain Python list

    # Safe int conversion — handles NaN gracefully
    def safe_int(val, default=0):
        try:
            return int(val) if pd.notna(val) else default
        except (ValueError, TypeError):
            return default

    def safe_float(val, default=0.0):
        try:
            return float(val) if pd.notna(val) else default
        except (ValueError, TypeError):
            return default

    def safe_str(val):
        return str(val).strip() if pd.notna(val) else None

    return {
        "video_id"          : safe_str(row["video_id"]),
        "country_code"      : safe_str(row["country_code"]),
        "title"             : safe_str(row["title"]),
        "channel_title"     : safe_str(row["channel_title"]),
        "category_name"     : safe_str(row["category_name"]),
        "thumbnail_link"    : safe_str(row.get("thumbnail_link")),
        "views"             : safe_int(row.get("views")),
        "likes"             : safe_int(row.get("likes")),
        "dislikes"          : safe_int(row.get("dislikes")),
        "comment_count"     : safe_int(row.get("comment_count")),
        "trending_date"     : safe_str(row.get("trending_date")),
        "publish_time"      : safe_str(row.get("publish_time")),
        "trending_frequency": safe_int(row.get("trending_frequency"), 1),
        "velocity_score"    : safe_float(row.get("velocity_score")),
        "tags"              : safe_str(row.get("tags")),
        "embedding"         : vec,
    }

# Quick sanity check on one record
sample = build_record(0)
print("Sample record keys :", list(sample.keys()))
print("Embedding length   :", len(sample["embedding"]))
print("video_id           :", sample["video_id"])
print("country_code       :", sample["country_code"])


Sample record keys : ['video_id', 'country_code', 'title', 'channel_title', 'category_name', 'thumbnail_link', 'views', 'likes', 'dislikes', 'comment_count', 'trending_date', 'publish_time', 'trending_frequency', 'velocity_score', 'tags', 'embedding']
Embedding length   : 1024
video_id           : WG8JK-cPu-g
country_code       : US


In [31]:
import time

BATCH_SIZE   = 25      # reduced from 100 — smaller batches beat the timeout
TOTAL        = len(full_df)
errors       = []
inserted     = 0
MAX_RETRIES  = 3

print(f"Starting upsert of {TOTAL:,} rows in batches of {BATCH_SIZE}…")
print(f"Estimated batches: {(TOTAL + BATCH_SIZE - 1) // BATCH_SIZE}\n")

for batch_start in range(0, TOTAL, BATCH_SIZE):
    batch_end     = min(batch_start + BATCH_SIZE, TOTAL)
    batch_records = [build_record(i) for i in range(batch_start, batch_end)]

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            supabase.table("videos").upsert(
                batch_records,
                on_conflict="video_id,country_code",
            ).execute()
            inserted += len(batch_records)
            break   # success — exit retry loop

        except Exception as e:
            if attempt < MAX_RETRIES:
                wait = attempt * 3   # 3s, 6s, 9s
                print(f"  ⚠ Batch {batch_start}–{batch_end} attempt {attempt} failed, retrying in {wait}s… ({e})")
                time.sleep(wait)
            else:
                errors.append({"batch_start": batch_start, "error": str(e)})
                print(f"  ✗ Batch {batch_start}–{batch_end} permanently failed: {e}")

    if batch_start % (BATCH_SIZE * 20) == 0:
        print(f"  ✓ {inserted:,} / {TOTAL:,} rows upserted")

    time.sleep(0.1)   # small breathing room between batches

print(f"\n{'='*50}")
print(f"Done. Upserted: {inserted:,} | Errors: {len(errors)}")
if errors:
    print("\nFailed batches (re-run from these row numbers):")
    for err in errors:
        print(f"  Row {err['batch_start']}: {err['error']}")

Starting upsert of 24,499 rows in batches of 25…
Estimated batches: 980

  ✓ 25 / 24,499 rows upserted
  ✓ 525 / 24,499 rows upserted
  ✓ 1,025 / 24,499 rows upserted
  ✓ 1,525 / 24,499 rows upserted
  ✓ 2,025 / 24,499 rows upserted
  ✓ 2,525 / 24,499 rows upserted
  ✓ 3,025 / 24,499 rows upserted
  ✓ 3,525 / 24,499 rows upserted
  ✓ 4,025 / 24,499 rows upserted
  ✓ 4,525 / 24,499 rows upserted
  ✓ 5,025 / 24,499 rows upserted
  ⚠ Batch 5275–5300 attempt 1 failed, retrying in 3s… ({'message': 'JSON could not be generated', 'code': 502, 'hint': 'Refer to full message for details', 'details': 'b\'<!DOCTYPE html>\\n<!--[if lt IE 7]> <html class="no-js ie6 oldie" lang="en-US"> <![endif]-->\\n<!--[if IE 7]>    <html class="no-js ie7 oldie" lang="en-US"> <![endif]-->\\n<!--[if IE 8]>    <html class="no-js ie8 oldie" lang="en-US"> <![endif]-->\\n<!--[if gt IE 8]><!--> <html class="no-js" lang="en-US"> <!--<![endif]-->\\n<head>\\n\\n<title> | 502: Bad gateway</title>\\n<meta charset="UTF-8" />

In [32]:
# ── Quick verification — counts per country in Supabase ──────────────────────
response = supabase.rpc("verify_population", {}).execute()     if False else None   # placeholder — use the query below instead

# Count rows per country via a simple select
result = (
    supabase.table("videos")
    .select("country_code", count="exact")
    .execute()
)
print(f"Total rows in Supabase: {result.count:,}")

# Per-country breakdown using raw SQL via rpc (if you create the function)
# Or just check Supabase dashboard → Table Editor → videos
print("\nCheck your Supabase dashboard → Table Editor → videos")
print("You should see ~24,499 rows across 10 countries.")

# Spot check — fetch one row back and confirm embedding dimension
spot = (
    supabase.table("videos")
    .select("video_id, country_code, title, velocity_score")
    .limit(3)
    .execute()
)
print("\nSpot check (first 3 rows):")
for row in spot.data:
    print(f"  [{row['country_code']}] {row['video_id']} | {row['title'][:50]} | velocity: {row['velocity_score']}")


Total rows in Supabase: 24,499

Check your Supabase dashboard → Table Editor → videos
You should see ~24,499 rows across 10 countries.

Spot check (first 3 rows):
  [US] WG8JK-cPu-g | At Home with Amy Sedaris - Amy's Not-So Holiday Sp | velocity: 0.8927
  [US] CV0J3Bq3BIc | Black Mirror - Black Museum | Official Trailer [HD | velocity: 1.5944
  [US] we2XODhx-CM | Chelsea Handler Questions Whether She's Cut Out fo | velocity: 1.5506
